# Aegis - Phase 7: StrongREJECT End-to-End ASR

Field-standard defense evaluation (Souly et al., "A StrongREJECT for Empty Jailbreaks", arXiv:2402.10260): measure whether an attack actually makes a victim LLM emit specific harmful content, judged by the StrongREJECT evaluator, comparing the victim ALONE against the victim placed behind Aegis - and do it per obfuscation channel, so the encoding axis Aegis targets is measured directly.

**Setup:** Accelerator -> GPU T4. Settings -> Internet: ON. Add Kaggle Secrets `HF_TOKEN` and `WANDB_API_KEY`. Then Run All.

In [ ]:
import sys, os, glob, subprocess
REPO_URL = "https://github.com/g25ait2149/aegis.git"     # same repo as the other notebooks
DEST = "/kaggle/working/aegis_src"
if os.path.isdir(os.path.join(DEST, ".git")):
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only"], check=False)   # already cloned -> update
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, DEST], check=False)
hits = glob.glob(DEST + "/**/aegis/__init__.py", recursive=True)
root = os.path.dirname(os.path.dirname(hits[0])) if hits else DEST
sys.path.insert(0, root)
for m in [m for m in sys.modules if m == "aegis" or m.startswith(("aegis.", "eval"))]:
    del sys.modules[m]
print("aegis repo at:", root)

## Secrets
Add as **Kaggle Secrets** (Add-ons -> Secrets):
- **`HF_TOKEN`** -> fits Aegis on the training corpus and loads the victim model. The StrongREJECT fine-tuned judge also needs this token to have access to the gated `google/gemma-2b` repo (accept its terms once on HuggingFace).
- **`WANDB_API_KEY`** -> logs the ASR table to **Weights & Biases** (project `aegis-llm-defense`).

In [ ]:
!pip -q install git+https://github.com/dsbowen/strong_reject.git@main accelerate
!pip -q uninstall -y torchao   # avoid a transformers/torchao version clash when loading the Gemma judge

## 1. End-to-end ASR: victim alone vs victim behind Aegis

`run_strongreject` loads the StrongREJECT forbidden-prompt set, generates a victim response once per prompt, then applies Aegis as a filter on top (`scan` -> block, `guard_turn` -> block). ASR is the mean StrongREJECT judge score (lower is safer); because Aegis only blocks, its ASR can never exceed the undefended ASR. Each obfuscation channel is its own row, so `char_spacing` (the current red-team gap) and `base64` etc. are visible side by side.

In [ ]:
from eval.strongreject_eval import run_strongreject

# Quick pass on the 60-prompt dev subset (fast). For the STANDARD result, use the full
# 313-prompt set: set full=True and drop n (see the note below).
rows = run_strongreject(
    victim_id="Qwen/Qwen2.5-1.5B-Instruct",   # small safety-tuned victim, fits a T4
    n=60,
    attacks=("identity", "base64", "char_spacing", "homoglyph", "zero_width", "roleplay_wrap"),
    judge_name="strongreject",                 # the real fine-tuned StrongREJECT judge
    wandb_log=True,
    # full=True,   # <- uncomment for the 313-prompt standard set (and remove n=60)
)
rows

## 2. Optional: the full cascade with the L2 guard

By default this scores L0-L1 plus the L4 output gate. Pass `use_guard=True` to attach your tuned L2 guard (the `aegis-rjd3-guard` adapter) so the harmful-topic channels that L1 alone waves through get a second look. Needs GPU headroom for victim + guard + judge; drop to a 0.5B victim if memory is tight. Uncomment the cell below to run it.

In [ ]:
# rows_cascade = run_strongreject(
#     victim_id="Qwen/Qwen2.5-1.5B-Instruct",
#     attacks=("identity", "base64", "char_spacing", "zero_width"),
#     judge_name="strongreject",
#     use_guard=True,        # attach the tuned L2 guard to the cascade
#     wandb_log=True,
# )
# rows_cascade

## 3. Guard-on-everything: harmful-topic coverage (`guard_mode="ensemble"`)

The section-2 cascade only consults the L2 guard on prompts L1 is *unsure* about; bare harmful questions score confidently-low on a jailbreak detector, so the cascade passes them **without asking the guard**. To measure harmful-topic coverage, run the guard on **every** (L0-normalized) input: `guard_mode="ensemble"` blocks when P(unsafe) >= `guard_threshold`.

**Important (verified on XSTest):** our tuned guard is a *jailbreak* detector - it flags a DAN prompt at 0.925 but bare harmful questions at ~0.01, so it does **not** cover harmful topics. Use `guard_impl="qwen3guard"` (a content-safety guard) for that axis; `guard_impl="tuned"` keeps the jailbreak guard. Uncomment to run.

In [ ]:
# Verified: guard_impl="tuned" is a JAILBREAK detector (misses bare harmful topics); use
# guard_impl="qwen3guard" (content-safety, runs on every L0-normalized input) for topic coverage.
# rows_ensemble = run_strongreject(
#     victim_id="Qwen/Qwen2.5-1.5B-Instruct",   # if OOM: Qwen/Qwen2.5-0.5B-Instruct
#     attacks=("identity", "base64", "char_spacing", "zero_width"),
#     judge_name="strongreject",
#     guard_mode="ensemble",
#     guard_impl="qwen3guard",   # content-safety L2 (use "tuned" for the jailbreak guard instead)
#     guard_threshold=0.5,
#     wandb_log=True,
# )
# rows_ensemble

## Next (Arc 1)
- Item 2: over-refusal on XSTest, alongside the in-house FRR.
- Item 3: Qwen3Guard-0.6B as a modern guard baseline.
- Item 4: AgentDojo (task utility and attack-success-rate jointly).